In [1]:
# Set seed for reproducibility
SEED = 45

# Import necessary libraries
import os

# Set environment variables before importing modules
# Set PYTHONHASHSEED for deterministic hash values
os.environ['PYTHONHASHSEED'] = str(SEED) 
# Set MPLCONFIGDIR to avoid creating config files in the home directory
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/' 

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python's 'random'
np.random.seed(SEED)
random.seed(SEED)

# --- PyTorch Setup and Device Configuration ---
import torch
torch.manual_seed(SEED)
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader

logs_dir = "tensorboard"

# 1. Check for CUDA (NVIDIA GPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    # Enable cuDNN benchmark for faster, but sometimes less reproducible, training
    # Set to False if absolute reproducibility is paramount
    torch.backends.cudnn.benchmark = True 
# 3. Default to CPU
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
# ---------------------------------------------

# Import other libraries
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot display settings (No more %matplotlib inline)
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)

2025-11-17 15:22:47.497561: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763392967.662385      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763392967.715684      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

PyTorch version: 2.6.0+cu124
Device: cuda


## 🔄 **Data Preprocessing**

In [2]:
# Load the dataset from a CSV file
df_train = pd.read_csv("/kaggle/input/pirate_pain_train.csv")
df_public_test = pd.read_csv("/kaggle/input/pirate_pain_test.csv")
df_labels = pd.read_csv("/kaggle/input/pirate_pain_train_labels.csv")

In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_labels['label_encoded'] = label_encoder.fit_transform(df_labels['label'])
# This gives: ['high_pain', 'low_pain', 'no_pain']
label_map = {'no_pain': 0, 'low_pain': 1, 'high_pain': 2}
df_labels['label_encoded'] = df_labels['label'].map(label_map)
df_labels.drop(["label"], axis =1)

,sample_index,label_encoded
0,0,0
1,1,0
2,2,1
3,3,0
4,4,0
...,...,...
656,656,0
657,657,0
658,658,0
659,659,0


In [4]:
#df = pd.concat([df_train, df_public_test], axis=0).reset_index(drop=True)

In [5]:
from sklearn.preprocessing import MinMaxScaler
def preProcess_with_2D_PE(df: pd.DataFrame, scaler: MinMaxScaler, fit_scaler: bool = True):
    
    PERIOD = 80

    # --- 1. Feature Engineering and Dropping Columns ---
    pain_cols = ["pain_survey_1", "pain_survey_2", "pain_survey_3", "pain_survey_4"]
    existing_pain_cols = [c for c in pain_cols if c in df.columns]
    
    if existing_pain_cols:
        df["pain_survey"] = np.floor(df[existing_pain_cols].median(axis=1)).astype(int)
        df.drop(columns=existing_pain_cols, inplace=True)

    # merged_n_features
    if all(col in df.columns for col in ['n_legs', 'n_hands', 'n_eyes']):
        df['merged_n_features'] = np.where(
            (df['n_legs'] == 'two') & (df['n_hands'] == 'two') & (df['n_eyes'] == 'two'),
            0, 
            1 
        )
        df.drop(columns=['n_legs', 'n_hands', 'n_eyes'], inplace=True)
    
    # Drop joint_30 if exists
    if 'joint_30' in df.columns:
        df.drop(columns=['joint_30'], inplace=True)
    
    # --- 2. 2D Sinusoidal Encoding ---
    if 'time' in df.columns:
        df["time_sin"] = np.sin(2 * np.pi * df["time"] / PERIOD)
        df["time_cos"] = np.cos(2 * np.pi * df["time"] / PERIOD)
        df.drop(columns=["time"], inplace=True)

    # --- 3. Scaling ---
    pe_cols = ["time_sin", "time_cos"]
    cols_to_scale = [col for col in df.columns if col not in ['sample_index', "label_encoded"] + pe_cols]
    
    if cols_to_scale:
        if fit_scaler:
            df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
        else:
            df[cols_to_scale] = scaler.transform(df[cols_to_scale])

feature_scaler = MinMaxScaler()
preProcess_with_2D_PE(df_train, feature_scaler, fit_scaler=True)
preProcess_with_2D_PE(df_public_test, feature_scaler, fit_scaler=False)

In [6]:
df_train.head()

,sample_index,joint_00,joint_01,joint_02,joint_03,joint_04,joint_05,joint_06,joint_07,joint_08,...,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,pain_survey,merged_n_features,time_sin,time_cos
0,0,0.777507,0.738252,0.779512,0.804419,0.714916,0.736643,0.639301,0.733981,0.478382,...,3.162813e-04,0.000004,0.014214,0.011376,0.018978,0.020291,0.5,0.0,0.000000,1.000000
1,0,0.806256,0.765147,0.761153,0.838021,0.735684,0.729533,0.654605,0.760554,0.486231,...,9.828599e-07,0.000000,0.010748,0.000000,0.009473,0.010006,1.0,0.0,0.078459,0.996917
2,0,0.767592,0.721439,0.772834,0.777832,0.724497,0.734962,0.692340,0.787647,0.441994,...,6.626013e-05,0.000003,0.013097,0.006830,0.017065,0.016856,1.0,0.0,0.156434,0.987688
3,0,0.666220,0.810416,0.763971,0.785928,0.679928,0.722504,0.589127,0.793524,0.469554,...,1.199337e-06,0.000000,0.009505,0.006274,0.020264,0.017981,1.0,0.0,0.233445,0.972370
4,0,0.774297,0.773366,0.772162,0.767017,0.747710,0.743156,0.677764,0.725437,0.477740,...,1.307199e-06,0.000007,0.004216,0.002132,0.023389,0.018477,1.0,0.0,0.309017,0.951057


In [7]:
#TIMESTAMP_COLS = ["time"]
#non_null_counts_per_user = df_train.groupby("sample_index")[TIMESTAMP_COLS].apply(
#    lambda x: x.notna().any().any()
#)
#users_with_all_nulls = non_null_counts_per_user[non_null_counts_per_user == False].index.tolist()

In [8]:
#total_non_nulls_per_user = df_train.groupby("sample_index")[TIMESTAMP_COLS].count().sum(axis=1)
#users_with_all_nulls_simple = total_non_nulls_per_user[total_non_nulls_per_user == 0].index.tolist()
#(users_with_all_nulls_simple, total_non_nulls_per_user)

In [9]:
#import pandas as pd
#import numpy as np

# --- Setup (Based on your context) ---
#USER_ID_COLUMN = 'sample_index' 
#TIMESTAMP_COLS = ["time"]
# Define a threshold for "low variance." A common starting point is 1e-6 (0.000001).
# You may need to adjust this based on the expected scale of your data.
#VARIANCE_THRESHOLD = 1000

# Assuming df_train is your DataFrame

# --- 1. Calculate Variance per User and Feature ---
# Group by user ID and calculate the variance for each timestamp column
#variance_per_user_feature = df_train.groupby("sample_index")[TIMESTAMP_COLS].var()

# --- 2. Check if Variance is Low for BOTH Features ---
# Create a boolean mask: True if the variance is BELOW the threshold.
#low_variance_mask = (variance_per_user_feature < VARIANCE_THRESHOLD)

# Check if BOTH 'time_sin' AND 'time_cos' are below the threshold for a given user.
# .all(axis=1) checks horizontally (across the features)
#users_with_low_variance_timestamps = low_variance_mask.all(axis=1)

# Get the list of user IDs that satisfy the condition
#users_to_flag = users_with_low_variance_timestamps[users_with_low_variance_timestamps == True].index.tolist()

# --- 3. Output Results ---
#print(f"Total unique users checked: {df_train[USER_ID_COLUMN].nunique()}")
#print(f"Variance check threshold: {VARIANCE_THRESHOLD}")
#print("---")

#if users_to_flag:
#    print(f"⚠️ Found {len(users_to_flag)} user(s) with low variance in ALL checked timestamp features:")
#    print(users_to_flag)
    
    # Optional: Display the actual variance for the first problematic user
#    first_flagged_user = users_to_flag[0]
#    print(f"\nExample Variance for User {first_flagged_user}:")
#    print(variance_per_user_feature.loc[first_flagged_user])
#else:
#    print("✅ No users found with variance below the threshold in both 'time_sin' and 'time_cos'.")

In [10]:
df_train["sample_index"]

0           0
1           0
2           0
3           0
4           0
         ... 
105755    660
105756    660
105757    660
105758    660
105759    660
Name: sample_index, Length: 105760, dtype: int64

In [11]:
from sklearn.model_selection import train_test_split

user_labels = df_labels.copy()

train_users, val_users = train_test_split(
    user_labels['sample_index'],
    test_size=0.2,  # 20% validation
    stratify=user_labels['label'], # This is the key for imbalance
    random_state=SEED
)

df_train_full = df_train.copy()

# Filter the full dataframe to get rows for train/val users
df_train_fold = df_train_full[df_train_full['sample_index'].isin(train_users)]
df_val_fold   = df_train_full[df_train_full['sample_index'].isin(val_users)]

# Merge labels back in
df_train_fold = df_train_fold.merge(user_labels[['sample_index', 'label', 'label_encoded']], on='sample_index', how='left')
df_val_fold   = df_val_fold.merge(user_labels[['sample_index', 'label', 'label_encoded']], on='sample_index', how='left')

df_train_fold = df_train_fold.drop(columns=['label'])
df_val_fold = df_val_fold.drop(columns=['label'])

In [12]:
# Features to float32
feature_cols_fold = [c for c in df_train_fold.columns if c not in ['sample_index', 'label_encoded']]
df_train_fold[feature_cols_fold] = df_train_fold[feature_cols_fold].astype('float32')
df_val_fold[feature_cols_fold] = df_val_fold[feature_cols_fold].astype('float32')

# Labels to int64
df_train_fold['label_encoded'] = df_train_fold['label_encoded'].astype('int64')
df_val_fold['label_encoded'] = df_val_fold['label_encoded'].astype('int64')

In [13]:
df_train_labeled = df_train.merge(
    user_labels[['sample_index', 'label', 'label_encoded']],
    on='sample_index',
    how='left'
)
df_train_labeled = df_train_labeled.drop(columns=["label"])

feature_cols_fold = [c for c in df_train_fold.columns if c not in ['sample_index', 'label_encoded']]

df_train_labeled[feature_cols_fold] = df_train_labeled[feature_cols_fold].astype('float32')

# Labels to int64
df_train_labeled['label_encoded'] = df_train_labeled['label_encoded'].astype('int64')
df_train_labeled.head()

,sample_index,joint_00,joint_01,joint_02,joint_03,joint_04,joint_05,joint_06,joint_07,joint_08,...,joint_25,joint_26,joint_27,joint_28,joint_29,pain_survey,merged_n_features,time_sin,time_cos,label_encoded
0,0,0.777507,0.738252,0.779512,0.804419,0.714916,0.736643,0.639301,0.733981,0.478382,...,0.000004,0.014214,0.011376,0.018978,0.020291,0.5,0.0,0.000000,1.000000,0
1,0,0.806256,0.765147,0.761153,0.838021,0.735684,0.729533,0.654604,0.760554,0.486231,...,0.000000,0.010748,0.000000,0.009473,0.010006,1.0,0.0,0.078459,0.996917,0
2,0,0.767592,0.721439,0.772834,0.777832,0.724497,0.734962,0.692340,0.787647,0.441994,...,0.000003,0.013097,0.006830,0.017065,0.016856,1.0,0.0,0.156434,0.987688,0
3,0,0.666220,0.810416,0.763971,0.785928,0.679928,0.722504,0.589127,0.793524,0.469554,...,0.000000,0.009505,0.006274,0.020264,0.017981,1.0,0.0,0.233445,0.972370,0
4,0,0.774297,0.773366,0.772162,0.767017,0.747710,0.743156,0.677764,0.725437,0.477740,...,0.000007,0.004216,0.002132,0.023389,0.018477,1.0,0.0,0.309017,0.951057,0


In [14]:
feature_cols = [col for col in df_train_fold.columns if 'joint_' in col]
feature_cols.extend(['pain_survey', 'merged_n_features'])


X_data_2d = df_train_fold[[col for col in df_train_fold.columns if col.startswith('joint_') or col in ['pain_survey', 'merged_n_features']]].values
Y_target_1d = df_train_fold['label_encoded'].values

# Check final count of X features:
num_features = X_data_2d.shape[1]

print(f"Raw X shape: {X_data_2d.shape}")
print(f"Raw Y shape: {Y_target_1d.shape}")

Raw X shape: (84480, 32)
Raw Y shape: (84480,)


### IMPORTANT
Now the CNN it's built, the CNN takes in input the time series with the slicing windows.
For this reason i guess that is possible to proceede as always and after that continue with the training.

## Prepare data for training

In [15]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [28]:
from typing import Optional, Tuple, List

def build_sequences(
    df: pd.DataFrame, 
    feature_cols: List[str], 
    id_col: str = 'sample_index', 
    label_col: Optional[str] = 'label_encoded', 
    window: int = 200, 
    stride: int = 200
) -> Tuple[np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Builds sequences from a time-series dataframe, optionally handling labels and
    always returning the sample_index for each generated window.
    
    Args:
        df (pd.DataFrame): The input DataFrame (e.g., df_train_fold).
        feature_cols (list): A list of column names to be used as features.
        id_col (str): The name of the column for unique sample IDs.
        label_col (Optional[str]): The name of the column for the labels. 
                                   Set to None if the DataFrame has no labels (e.g., test set).
        window (int): The size of each sequence (window).
        stride (int): The step size between sequences.
        
    Returns:
        sequences (np.ndarray): The 3D array of sequences (N_sequences, window, N_features).
        labels (Optional[np.ndarray]): The 1D array of labels, or None if label_col is None.
        sample_ids (np.ndarray): The 1D array of original sample IDs for each window.
    """
    num_features = len(feature_cols)
    dataset = []
    labels = []
    sample_ids = [] # NEW: List to collect the sample ID for each window
    
    # Check if we should extract labels
    extract_labels = (label_col is not None) and (label_col in df.columns)

    # Iterate over unique sample IDs
    for sample_id in df[id_col].unique():
        
        # Get the dataframe for the current sample
        temp_df = df[df[id_col] == sample_id]

        # Extract feature data for the current ID
        temp_features = temp_df[feature_cols].values

        if extract_labels:
            # Retrieve the single label for the current ID and convert to int
            label = int(temp_df[label_col].iloc[0])

        # Calculate padding length to ensure full windows (if not using the sliding window end condition)
        # Note: Your original code uses padding. If your final pipeline uses standard sliding
        # window logic (range(0, total_steps - window + 1, stride)) without padding, 
        # you might remove this padding block for consistency. Assuming you keep it for now.
        padding_len = (window - len(temp_features) % window) % window
        
        if padding_len > 0:
            # Create zero padding and concatenate with the data
            padding = np.zeros((padding_len, num_features), dtype='float32')
            temp_features = np.concatenate((temp_features, padding))

        # Build feature windows and associate them with labels and IDs
        idx = 0
        while idx + window <= len(temp_features):
            dataset.append(temp_features[idx:idx + window])
            
            # CRITICAL: Append the sample ID for this window
            sample_ids.append(sample_id) 
            
            if extract_labels:
                labels.append(label)
            
            idx += stride

    # Convert lists to numpy arrays for further processing
    sequences = np.array(dataset, dtype=np.float32)
    sample_ids_array = np.array(sample_ids, dtype=df[id_col].dtype) # Use appropriate dtype

    if extract_labels:
        labels_array = np.array(labels, dtype=np.int64)
        # Return 3 elements: sequences, labels, sample_ids
        return sequences, labels_array, sample_ids_array
    else:
        # Return 3 elements: sequences, None, sample_ids
        return sequences, None, sample_ids_array

## 🛠️ **Model Building**

In [17]:
def recurrent_summary(model, input_size):
    """
    Custom summary function that emulates torchinfo's output while correctly
    counting parameters for RNN/GRU/LSTM layers.

    This function is designed for models whose direct children are
    nn.Linear, nn.RNN, nn.GRU, or nn.LSTM layers.

    Args:
        model (nn.Module): The model to analyze.
        input_size (tuple): Shape of the input tensor (e.g., (seq_len, features)).
    """

    # Dictionary to store output shapes captured by forward hooks
    output_shapes = {}
    # List to track hook handles for later removal
    hooks = []

    def get_hook(name):
        """Factory function to create a forward hook for a specific module."""
        def hook(module, input, output):
            # Handle RNN layer outputs (returns a tuple)
            if isinstance(output, tuple):
                # output[0]: all hidden states with shape (batch, seq_len, hidden*directions)
                shape1 = list(output[0].shape)
                shape1[0] = -1  # Replace batch dimension with -1

                # output[1]: final hidden state h_n (or tuple (h_n, c_n) for LSTM)
                if isinstance(output[1], tuple):  # LSTM case: (h_n, c_n)
                    shape2 = list(output[1][0].shape)  # Extract h_n only
                else:  # RNN/GRU case: h_n only
                    shape2 = list(output[1].shape)

                # Replace batch dimension (middle position) with -1
                shape2[1] = -1

                output_shapes[name] = f"[{shape1}, {shape2}]"

            # Handle standard layer outputs (e.g., Linear)
            else:
                shape = list(output.shape)
                shape[0] = -1  # Replace batch dimension with -1
                output_shapes[name] = f"{shape}"
        return hook

    # 1. Determine the device where model parameters reside
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")  # Fallback for models without parameters

    # 2. Create a dummy input tensor with batch_size=1
    dummy_input = torch.randn(1, *input_size).to(device)

    # 3. Register forward hooks on target layers
    # Iterate through direct children of the model (e.g., self.rnn, self.classifier)
    for name, module in model.named_children():
        if isinstance(module, (nn.Linear, nn.RNN, nn.GRU, nn.LSTM)):
            # Register the hook and store its handle for cleanup
            hook_handle = module.register_forward_hook(get_hook(name))
            hooks.append(hook_handle)

    # 4. Execute a dummy forward pass in evaluation mode
    model.eval()
    with torch.no_grad():
        try:
            model(dummy_input)
        except Exception as e:
            print(f"Error during dummy forward pass: {e}")
            # Clean up hooks even if an error occurs
            for h in hooks:
                h.remove()
            return

    # 5. Remove all registered hooks
    for h in hooks:
        h.remove()

    # --- 6. Print the summary table ---

    print("-" * 79)
    # Column headers
    print(f"{'Layer (type)':<25} {'Output Shape':<28} {'Param #':<18}")
    print("=" * 79)

    total_params = 0
    total_trainable_params = 0

    # Iterate through modules again to collect and display parameter information
    for name, module in model.named_children():
        if name in output_shapes:
            # Count total and trainable parameters for this module
            module_params = sum(p.numel() for p in module.parameters())
            trainable_params = sum(p.numel() for p in module.parameters() if p.requires_grad)

            total_params += module_params
            total_trainable_params += trainable_params

            # Format strings for display
            layer_name = f"{name} ({type(module).__name__})"
            output_shape_str = str(output_shapes[name])
            params_str = f"{trainable_params:,}"

            print(f"{layer_name:<25} {output_shape_str:<28} {params_str:<15}")

    print("=" * 79)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {total_trainable_params:,}")
    print(f"Non-trainable params: {total_params - total_trainable_params:,}")
    print("-" * 79)

In [18]:
class CSHN_ConvBlock(nn.Module):
    """ Implements the CSHN Core: Conv1D -> BatchNorm1D -> LeakyReLU. """
    def __init__(self, in_channels, out_channels, kernel_size, padding):
        super(CSHN_ConvBlock, self).__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = nn.SiLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.act(x)
        return x

class CSHN_FeatureExtractor(nn.Module):
    """ 3-layer CNN feature extractor with tunable filter sizes. """
    def __init__(self, num_raw_features, c1_filters, c2_filters, c3_filters, cnn_dropout_rate):
        super(CSHN_FeatureExtractor, self).__init__()
        
        # Fixed Kernel/Padding based on your architecture
        C1_KERNEL, C2_KERNEL, C3_KERNEL = 8, 5, 3 
        C2_PADDING, C3_PADDING = 2, 1
        
        # C1 (Input features -> C1_FILTERS)
        self.c1 = CSHN_ConvBlock(num_raw_features, c1_filters, C1_KERNEL, padding=0)
        # C2 (C1_FILTERS -> C2_FILTERS)
        self.c2 = CSHN_ConvBlock(c1_filters, c2_filters, C2_KERNEL, padding=C2_PADDING)  
        # C3 (C2_FILTERS -> C3_FILTERS)
        self.c3 = CSHN_ConvBlock(c2_filters, c3_filters, C3_KERNEL, padding=C3_PADDING)
        
        self.dropout = nn.Dropout(cnn_dropout_rate)

    def forward(self, x):
        # x enters as (Batch, Time, Features) -> Permute to (Batch, Features, Time) for Conv1D
        x = x.transpose(1, 2)
        
        x = self.c1(x) 
        x = self.c2(x) 
        x = self.c3(x) 
        
        # Permute back for the RNN input: (Batch, Features, Time) -> (Batch, Time, Features)
        x = x.transpose(1, 2)
        
        x = self.dropout(x)
        return x

In [19]:
class RecurrentClassifier(nn.Module):
    """
    Generic RNN classifier (RNN, LSTM, GRU).
    Uses the last hidden state for classification.
    """
    def __init__(
            self,
            input_size,
            hidden_size,
            num_layers,
            num_classes,
            rnn_type='GRU',        # 'RNN', 'LSTM', or 'GRU'
            bidirectional=False,
            dropout_rate=0.2
            ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

        # Map string name to PyTorch RNN class
        rnn_map = {
            'RNN': nn.RNN,
            'LSTM': nn.LSTM,
            'GRU': nn.GRU
        }

        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', or 'GRU'")

        rnn_module = rnn_map[rnn_type]

        # Dropout is only applied between layers (if num_layers > 1)
        dropout_val = dropout_rate if num_layers > 1 else 0

        # Create the recurrent layer
        self.rnn = rnn_module(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,       # Input shape: (batch, seq_len, features)
            bidirectional=bidirectional,
            dropout=dropout_val
        )

        # Calculate input size for the final classifier
        if self.bidirectional:
            classifier_input_size = hidden_size * 2 # Concat fwd + bwd
        else:
            classifier_input_size = hidden_size

        # Final classification layer
        self.classifier = nn.Linear(classifier_input_size, num_classes)

    def forward(self, x):
        """
        x shape: (batch_size, seq_length, input_size)
        """

        # rnn_out shape: (batch_size, seq_len, hidden_size * num_directions)
        rnn_out, hidden = self.rnn(x)

        # LSTM returns (h_n, c_n), we only need h_n
        if self.rnn_type == 'LSTM':
            hidden = hidden[0]

        # hidden shape: (num_layers * num_directions, batch_size, hidden_size)

        if self.bidirectional:
            # Reshape to (num_layers, 2, batch_size, hidden_size)
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)

            # Concat last fwd (hidden[-1, 0, ...]) and bwd (hidden[-1, 1, ...])
            # Final shape: (batch_size, hidden_size * 2)
            hidden_to_classify = torch.cat([hidden[-1, 0, :, :], hidden[-1, 1, :, :]], dim=1)
        else:
            # Take the last layer's hidden state
            # Final shape: (batch_size, hidden_size)
            hidden_to_classify = hidden[-1]

        # Get logits
        logits = self.classifier(hidden_to_classify)
        return logits

In [20]:
class CSHN_HybridClassifier(nn.Module):
    """ Combines the CNN feature extractor and the RNN classification head. """
    def __init__(self, cnn_params: dict, rnn_params: dict, num_raw_features: int, num_classes: int):
        super(CSHN_HybridClassifier, self).__init__()
        
        # 1. Feature Extractor (CNN)
        self.cnn = CSHN_FeatureExtractor(
            num_raw_features=num_raw_features,
            c1_filters=cnn_params['c1_filters'],
            c2_filters=cnn_params['c2_filters'],
            c3_filters=cnn_params['c3_filters'],
            cnn_dropout_rate=cnn_params['cnn_dropout']
        )
        
        # The input size for the RNN is the output channel count of the final CNN layer (C3_FILTERS)
        rnn_input_size = cnn_params['c3_filters']
        
        # 2. Recurrent Classification Head (RNN)
        self.rnn = RecurrentClassifier(
            input_size=rnn_input_size,
            hidden_size=rnn_params['hidden_size'],
            num_layers=rnn_params['num_layers'],
            num_classes=num_classes,
            dropout_rate=rnn_params['rnn_dropout'],
            bidirectional=rnn_params['bidirectional'],
            rnn_type=rnn_params['rnn_type']
        )
        
    def forward(self, x):
        # x shape: (N, T_in, F_in)
        x = self.cnn(x)
        # x shape after CNN: (N, T_out, F_out) - This is the sequence for the RNN
        x = self.rnn(x)
        # x shape after RNN: (N, num_classes)
        return x

In [21]:
def initialize_weights(model: nn.Module, init_scheme: str):
    """
    Initializes weights, correctly skipping 1D tensors (like biases) 
    that cause 'Fan in and fan out' errors.
    """
    for name, param in model.named_parameters():
        # Skip parameters with fewer than 2 dimensions (typically biases and BatchNorm params)
        if param.dim() < 2:
            # Optionally initialize biases to zero, or skip them
            if 'bias' in name:
                torch.nn.init.constant_(param.data, 0.0)
            continue

        # Apply initialization schemes only to weight matrices (dim >= 2)
        if init_scheme == 'xavier_uniform':
            torch.nn.init.xavier_uniform_(param.data)
        elif init_scheme == 'kaiming_normal':
            # Good for ReLU/Leaky ReLU (common in surrounding layers)
            torch.nn.init.kaiming_normal_(param.data, mode='fan_in', nonlinearity='leaky_relu')
        elif init_scheme == 'orthogonal':
            torch.nn.init.orthogonal_(param.data)


In [22]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
    """
    Perform one complete training epoch through the entire training dataset.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): Lambda for L1 regularization
        l2_lambda (float): Lambda for L2 regularization

    Returns:
        tuple: (average_loss, f1 score) - Training loss and f1 score for this epoch
    """
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs, targets = inputs.to(device), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [23]:
def validate_one_epoch(model, val_loader, criterion, device):
    """
    Perform one complete validation epoch through the entire validation dataset.

    Args:
        model (nn.Module): The neural network model to evaluate (must be in eval mode)
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        criterion (nn.Module): Loss function used to calculate validation loss
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)

    Returns:
        tuple: (average_loss, accuracy) - Validation loss and accuracy for this epoch

    Note:
        This function automatically sets the model to evaluation mode and disables
        gradient computation for efficiency during validation.
    """
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs, targets in val_loader:
            # Move data to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [24]:
def log_metrics_to_tensorboard(writer, epoch, train_loss, train_f1, val_loss, val_f1, model):
    """
    Log training metrics and model parameters to TensorBoard for visualization.

    Args:
        writer (SummaryWriter): TensorBoard SummaryWriter object for logging
        epoch (int): Current epoch number (used as x-axis in TensorBoard plots)
        train_loss (float): Training loss for this epoch
        train_f1 (float): Training f1 score for this epoch
        val_loss (float): Validation loss for this epoch
        val_f1 (float): Validation f1 score for this epoch
        model (nn.Module): The neural network model (for logging weights/gradients)

    Note:
        This function logs scalar metrics (loss/f1 score) and histograms of model
        parameters and gradients, which helps monitor training progress and detect
        issues like vanishing/exploding gradients.
    """
    # Log scalar metrics
    writer.add_scalar('Loss/Training', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('F1/Training', train_f1, epoch)
    writer.add_scalar('F1/Validation', val_f1, epoch)

    # Log model parameters and gradients
    for name, param in model.named_parameters():
        if param.requires_grad:
            # Check if the tensor is not empty before adding a histogram
            if param.numel() > 0:
                writer.add_histogram(f'{name}/weights', param.data, epoch)
            if param.grad is not None:
                # Check if the gradient tensor is not empty before adding a histogram
                if param.grad.numel() > 0:
                    if param.grad is not None and torch.isfinite(param.grad).all():
                        writer.add_histogram(f'{name}/gradients', param.grad.data, epoch)

In [25]:
import torch
import torch.nn as nn
import optuna
# Importa la funzione di utilità se necessario (anche se lo faremo manualmente) 7
# import torch.nn.utils as nn_utils 

def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
        restore_best_weights=True, writer=None, verbose=10, experiment_name="",
        trial=None, scheduler=None, weight_max_norm=None):
    """
    Train the neural network model on the training data and validate on the validation data.
    """

    # --- NUOVA FUNZIONE: Normalizzazione/Vincolo dei Pesi ---
    def apply_weight_constraint(model, max_norm):
        with torch.no_grad():
            for name, module in model.named_modules():
                # Applica solo ai layer con pesi (es. Linear, Conv1d, Conv2d)
                if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d)):
                    # Vincola il parametro 'weight'
                    if hasattr(module, 'weight') and module.weight is not None:
                        # Calcola la norma L2 del tensore dei pesi
                        norm = module.weight.norm(2)
                        
                        if norm > max_norm:
                            # Se la norma è maggiore del vincolo, riscala il peso.
                            # Ciò garantisce che la norma L2 sia esattamente 'max_norm' o inferiore.
                            module.weight.data.mul_(max_norm / norm)

    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    # Initialize best_metric before loop
    best_metric = float('-inf') if mode == 'max' else float('inf')
    best_epoch = 0
    
    if patience > 0:
        patience_counter = 0

    if verbose > 0: # Print only if verbose
        print(f"Training {epochs} epochs...")

    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):

        # Forward pass, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )
        
        if weight_max_norm is not None and weight_max_norm > 0:
            apply_weight_constraint(model, weight_max_norm)

        # Evaluate model on validation data
        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Store metrics
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        # Write to TensorBoard
        if writer is not None:
            log_metrics_to_tensorboard(
                writer, epoch, train_loss, train_f1, val_loss, val_f1, model
            )

        # Print progress
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                      f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                      f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")

        # Get current metric for Pruning & Early Stopping
        current_metric = training_history[evaluation_metric][-1]

        # Scheduler Step
        if scheduler is not None:
            scheduler.step(current_metric)
        
        # Optuna Pruning
        if trial is not None:
            try:
                trial.report(current_metric, epoch)
            except Exception as e:
                # Gestione dell'errore (solo se Optuna è effettivamente usato)
                print(f"Error during Optuna report: {e}") 
                pass

            if trial.should_prune():
                # Pruning requested
                raise optuna.exceptions.TrialPruned()
        # End Pruning

        # Early stopping logic (omesso per brevità, resta invariato)
        if patience > 0:
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                # Non uso la variabile `experiment_name` qui, assumo che sia definita in un contesto più ampio
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    if verbose > 0:
                        print(f"Early stopping triggered after {epoch} epochs.")
                    break

    # Restore best model weights (omesso per brevità, resta invariato)
    if restore_best_weights and patience > 0 and best_epoch > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        if verbose > 0:
            print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping (omesso per brevità, resta invariato)
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
        if mode == 'max':
            best_metric = max(training_history[evaluation_metric])
        else:
            best_metric = min(training_history[evaluation_metric])
            
    if patience > 0 and best_epoch == 0:
         if mode == 'max':
             best_metric = max(training_history[evaluation_metric])
         else:
             best_metric = min(training_history[evaluation_metric])


    # Close TensorBoard writer
    if writer is not None:
        writer.close()

    return model, training_history, best_metric

In [26]:
df_train_fold.head()

,sample_index,joint_00,joint_01,joint_02,joint_03,joint_04,joint_05,joint_06,joint_07,joint_08,...,joint_25,joint_26,joint_27,joint_28,joint_29,pain_survey,merged_n_features,time_sin,time_cos,label_encoded
0,0,0.777507,0.738252,0.779512,0.804419,0.714916,0.736643,0.639301,0.733981,0.478382,...,0.000004,0.014214,0.011376,0.018978,0.020291,0.5,0.0,0.000000,1.000000,0
1,0,0.806256,0.765147,0.761153,0.838021,0.735684,0.729533,0.654604,0.760554,0.486231,...,0.000000,0.010748,0.000000,0.009473,0.010006,1.0,0.0,0.078459,0.996917,0
2,0,0.767592,0.721439,0.772834,0.777832,0.724497,0.734962,0.692340,0.787647,0.441994,...,0.000003,0.013097,0.006830,0.017065,0.016856,1.0,0.0,0.156434,0.987688,0
3,0,0.666220,0.810416,0.763971,0.785928,0.679928,0.722504,0.589127,0.793524,0.469554,...,0.000000,0.009505,0.006274,0.020264,0.017981,1.0,0.0,0.233445,0.972370,0
4,0,0.774297,0.773366,0.772162,0.767017,0.747710,0.743156,0.677764,0.725437,0.477740,...,0.000007,0.004216,0.002132,0.023389,0.018477,1.0,0.0,0.309017,0.951057,0


## Optuna

In [ ]:
import pandas as pd
import numpy as np
import os
import random # Added import for set_seed
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from sklearn.model_selection import StratifiedKFold # Replaced GroupKFold/SGKF
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import MinMaxScaler
import optuna

# NOTE: The functions 'preProcess_with_2D_PE', 'build_sequences', 'make_loader',
# 'CSHN_HybridClassifier', 'initialize_weights', 'fit', and 'device' are assumed
# to be defined elsewhere and available in the execution environment.

BATCH_SIZE = 64
EPOCHS = 200
PATIENCE = 15
N_SPLITS = 3
SEED = 42 # Assuming a global SEED variable is defined

def set_seed(seed_value):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)

# --- OOF placeholders for training fold only ---
# NOTE: The length should be the number of UNIQUE samples/groups in df_train_fold,
# not the number of rows, as OOF is aggregated per sample_index.
num_classes = 3
unique_samples_in_train = df_train_labeled['sample_index'].nunique()
oof_preds_train = np.zeros(unique_samples_in_train, dtype=np.int64)
oof_probs_train = np.zeros((unique_samples_in_train, num_classes), dtype=np.float32)

# --- Stratified Group K-Fold Preparation (USING SKLEARN) ---

# 1. Create a unique label for each group based on the classes it contains.
group_level_labels_df = df_train_labeled.groupby('sample_index')['label_encoded'].agg(
    lambda x: tuple(sorted(x.unique()))
).reset_index()

# 2. Encode the unique class tuples into integers for StratifiedKFold
group_labels_encoded = pd.Categorical(group_level_labels_df['label_encoded']).codes

# 3. Extract the arrays needed for SKF split
sample_indices = group_level_labels_df['sample_index'].values # Array of unique user IDs
data_labels_for_stratification = group_labels_encoded # Array of encoded group labels

# 4. Initialize StratifiedKFold (from sklearn)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


# --- Optuna objective ---
def objective(trial):
    fold_scores = []

    # Trial-level hyperparameters
    seed_hp = trial.suggest_int("SEED", 0, 10000)
    set_seed(seed_hp)

    window_hp = trial.suggest_categorical("WINDOW", [20, 40, 60])
    stride_hp = trial.suggest_categorical("STRIDE", [5, 10, 15, 20, 30])
    if stride_hp >= window_hp:
        raise optuna.exceptions.TrialPruned()

    c1_filters_hp = trial.suggest_categorical("c1_filters", [32])
    c2_filters_hp = trial.suggest_categorical("c2_filters", [64])
    c3_filters_hp = trial.suggest_categorical("c3_filters", [128])

    cnn_dropout_hp = trial.suggest_float("cnn_dropout", 0.0, 0.5)

    hidden_size_hp = trial.suggest_categorical("hidden_size", [16, 32, 64, 128])
    num_layers_hp = trial.suggest_categorical("num_layers", [1, 2, 3])
    rnn_dropout_hp = trial.suggest_float("rnn_dropout", 0.1, 0.7)
    rnn_type_hp = trial.suggest_categorical("rnn_type", ["GRU"])
    bidirectional_hp = trial.suggest_categorical("bidirectional", [True, False])
    init_scheme_hp = trial.suggest_categorical("init_scheme", ["xavier_uniform", "orthogonal", "kaiming_normal"])
    lr_hp = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay_hp = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    l1_lambda_hp = trial.suggest_float("l1_lambda", 1e-7, 1e-4, log=True)
    weight_max_norm_hp = trial.suggest_float("weight_max_norm", 0.1, 6.0)

    # --- K-Fold CV on training fold (MODIFIED SPLIT) ---
    # We use skf.split on the group indices and labels for stratification
    for fold, (train_group_idx, val_group_idx) in enumerate(
        skf.split(
            X=sample_indices, # Split the array of unique user IDs
            y=data_labels_for_stratification # Stratify based on the encoded group labels
        )
    ):
        # 1. Map the indices back to the actual user IDs (sample_index)
        train_groups = sample_indices[train_group_idx]
        val_groups = sample_indices[val_group_idx]

        # 2. Filter the main DataFrame (df_train_labeled) based on the selected group IDs
        # This replaces the GroupKFold indexing and achieves no-leakage group splitting.
        df_train_k = df_train_labeled[df_train_labeled['sample_index'].isin(train_groups)].reset_index(drop=True)
        df_val_k   = df_train_labeled[df_train_labeled['sample_index'].isin(val_groups)].reset_index(drop=True)

        # --- Preprocess with 2D positional encoding ---
        scaler = MinMaxScaler()
        preProcess_with_2D_PE(df_train_k, scaler, fit_scaler=True)
        preProcess_with_2D_PE(df_val_k, scaler, fit_scaler=False)

        # Convert types after preprocessing
        feature_cols_fold = [c for c in df_train_k.columns if c not in ['sample_index', 'label_encoded']]
        df_train_k[feature_cols_fold] = df_train_k[feature_cols_fold].astype('float32')
        df_val_k[feature_cols_fold] = df_val_k[feature_cols_fold].astype('float32')

        # --- Build sequences (ASSUMING build_sequences RETURNS (X, y, sample_ids)) ---
        # The sample IDs are crucial for OOF aggregation
        X_train, y_train, _ = build_sequences(
            df_train_k, feature_cols_fold, 'sample_index', 'label_encoded', window_hp, stride_hp
        )
        X_val, y_val, val_sequence_sample_ids = build_sequences(
            df_val_k, feature_cols_fold, 'sample_index', 'label_encoded', window_hp, stride_hp
        )

        X_train = X_train.astype(np.float32)
        X_val = X_val.astype(np.float32)
        y_train = y_train.astype(np.int64)
        y_val = y_val.astype(np.int64)

        # --- Compute class weights ---
        labels_in_fold = np.unique(y_train)
        class_weights_present = compute_class_weight('balanced', classes=labels_in_fold, y=y_train)
        weights_tensor_np = np.zeros(num_classes, dtype=np.float32)
        for i, label in enumerate(labels_in_fold):
            weights_tensor_np[label] = class_weights_present[i]
        weights_tensor = torch.tensor(weights_tensor_np, dtype=torch.float32).to(device)

        # --- Create DataLoaders ---
        train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
        val_ds   = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
        train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
        val_loader   = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

        # --- Model ---
        cnn_params = {'c1_filters': c1_filters_hp, 'c2_filters': c2_filters_hp, 'c3_filters': c3_filters_hp, 'cnn_dropout': cnn_dropout_hp}
        rnn_params = {'hidden_size': hidden_size_hp, 'num_layers': num_layers_hp, 'rnn_dropout': rnn_dropout_hp, 'bidirectional': bidirectional_hp, 'rnn_type': rnn_type_hp}
        model = CSHN_HybridClassifier(cnn_params=cnn_params, rnn_params=rnn_params, num_raw_features=len(feature_cols_fold), num_classes=num_classes).to(device)
        initialize_weights(model, init_scheme=init_scheme_hp)

        criterion = nn.CrossEntropyLoss(weight=weights_tensor)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr_hp, weight_decay=weight_decay_hp)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=10, min_lr=1e-7)
        scaler_amp = torch.amp.GradScaler(enabled=(device.type=='cuda'))

        os.makedirs("models", exist_ok=True)

        # --- Train ---
        try:
            _, _, best_val_f1 = fit(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                epochs=EPOCHS,
                criterion=criterion,
                optimizer=optimizer,
                scaler=scaler_amp,
                device=device,
                l1_lambda=l1_lambda_hp,
                l2_lambda=0,
                patience=PATIENCE,
                evaluation_metric="val_f1",
                mode='max',
                restore_best_weights=True,
                trial=trial,
                scheduler=scheduler,
                weight_max_norm=weight_max_norm_hp,
                writer=None,
                verbose=1,
                experiment_name=f"optuna_trial_{trial.number}_fold_{fold}"
            )

            # --- OOF aggregation per sample_index ---
            model.eval()
            probs_list = []
            with torch.no_grad():
                for xb, _ in val_loader:
                    xb = xb.to(device)
                    out = model(xb)
                    probs_list.append(out.softmax(dim=1).cpu().numpy())

                probs_fold = np.concatenate(probs_list, axis=0)

                # Use the sample IDs returned by build_sequences
                current_val_sample_ids = val_sequence_sample_ids[:probs_fold.shape[0]]

                probs_fold_df = pd.DataFrame(probs_fold, columns=range(num_classes))
                probs_fold_df['sample_index'] = current_val_sample_ids

                # Aggregate probabilities by averaging across windows for each sample_index
                probs_fold_avg_df = probs_fold_df.groupby('sample_index')[list(range(num_classes))].mean()
                probs_fold_avg = probs_fold_avg_df.values

                unique_val_sample_ids = probs_fold_avg_df.index.values

                # --- OOF update logic (Robust Mapping) ---
                # We need a dictionary that maps the sample_index ID to the row index of the global OOF array.
                global_sample_index_list = df_train_labeled['sample_index'].unique()
                global_sample_index_map = {id: i for i, id in enumerate(global_sample_index_list)}

                # Get the OOF indices that correspond to the unique samples in this validation fold
                oof_indices_to_update = [global_sample_index_map[id] for id in unique_val_sample_ids]

                oof_probs_train[oof_indices_to_update] = probs_fold_avg
                oof_preds_train[oof_indices_to_update] = probs_fold_avg.argmax(axis=1)

            fold_scores.append(best_val_f1)

        except optuna.exceptions.TrialPruned:
            raise
        except Exception as e:
            print(f"Trial {trial.number} Fold {fold} failed: {e}")
            return 0.0

    return np.mean(fold_scores)

# --- Run Study (Assuming Optuna is available) ---
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=5, n_startup_trials=3))
print("Starting Optuna optimization with Stratified Group K-Fold (using sklearn)...")
study.optimize(objective, n_trials=50)

print("\n--- Optuna Tuning Complete ---")
print(f"Best trial: {study.best_trial.number}, Best mean F1 (training folds): {study.best_value:.4f}")
print("Best hyperparameters:")
for k,v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2025-11-17 15:24:34,675] A new study created in memory with name: no-name-08f1c701-d9cb-477a-b7f5-04c21d9247d9


Starting Optuna optimization with Stratified Group K-Fold (using sklearn)...
Training 200 epochs...
Epoch   1/200 | Train: Loss=0.7531, F1 Score=0.7519 | Val: Loss=1.0283, F1 Score=0.6751
Epoch   2/200 | Train: Loss=0.6705, F1 Score=0.8345 | Val: Loss=1.0337, F1 Score=0.6972
Epoch   3/200 | Train: Loss=0.5820, F1 Score=0.8575 | Val: Loss=1.0525, F1 Score=0.6751
Epoch   4/200 | Train: Loss=0.6156, F1 Score=0.8377 | Val: Loss=1.0503, F1 Score=0.6751
Epoch   5/200 | Train: Loss=0.5787, F1 Score=0.8577 | Val: Loss=1.0565, F1 Score=0.6751
Epoch   6/200 | Train: Loss=0.5401, F1 Score=0.8668 | Val: Loss=1.0505, F1 Score=0.6751
Epoch   7/200 | Train: Loss=0.6003, F1 Score=0.8430 | Val: Loss=1.0670, F1 Score=0.6751
Epoch   8/200 | Train: Loss=0.5985, F1 Score=0.8489 | Val: Loss=1.0763, F1 Score=0.6751
Epoch   9/200 | Train: Loss=0.5486, F1 Score=0.8465 | Val: Loss=1.0704, F1 Score=0.6751
Epoch  10/200 | Train: Loss=0.5790, F1 Score=0.8545 | Val: Loss=1.0610, F1 Score=0.6751
Epoch  11/200 | Trai

[I 2025-11-17 15:35:11,160] Trial 0 finished with value: 0.9224502800789871 and parameters: {'SEED': 9938, 'WINDOW': 60, 'STRIDE': 5, 'c1_filters': 32, 'c2_filters': 64, 'c3_filters': 128, 'cnn_dropout': 0.26814738644088254, 'hidden_size': 16, 'num_layers': 3, 'rnn_dropout': 0.4098053587862246, 'rnn_type': 'GRU', 'bidirectional': False, 'init_scheme': 'xavier_uniform', 'lr': 0.0006801009335646962, 'weight_decay': 0.0003168428972529413, 'l1_lambda': 1.257438841741642e-06, 'weight_max_norm': 0.4813016928754955}. Best is trial 0 with value: 0.9224502800789871.


Training 200 epochs...
Epoch   1/200 | Train: Loss=1.6014, F1 Score=0.2706 | Val: Loss=1.1170, F1 Score=0.6751
Epoch   2/200 | Train: Loss=1.2349, F1 Score=0.5093 | Val: Loss=0.9258, F1 Score=0.6739
Epoch   3/200 | Train: Loss=1.0798, F1 Score=0.7100 | Val: Loss=0.8619, F1 Score=0.7523
Epoch   4/200 | Train: Loss=0.9946, F1 Score=0.7633 | Val: Loss=0.8010, F1 Score=0.7726
Epoch   5/200 | Train: Loss=0.9330, F1 Score=0.8012 | Val: Loss=0.7551, F1 Score=0.7897
Epoch   6/200 | Train: Loss=0.8864, F1 Score=0.8164 | Val: Loss=0.7101, F1 Score=0.8212
Epoch   7/200 | Train: Loss=0.8376, F1 Score=0.8366 | Val: Loss=0.6733, F1 Score=0.8333
Epoch   8/200 | Train: Loss=0.8070, F1 Score=0.8460 | Val: Loss=0.6684, F1 Score=0.8193
Epoch   9/200 | Train: Loss=0.7746, F1 Score=0.8427 | Val: Loss=0.6526, F1 Score=0.8262
Epoch  10/200 | Train: Loss=0.7370, F1 Score=0.8690 | Val: Loss=0.6297, F1 Score=0.8483
Epoch  11/200 | Train: Loss=0.7095, F1 Score=0.8717 | Val: Loss=0.6383, F1 Score=0.8193
Epoch  12

[I 2025-11-17 15:40:18,316] Trial 1 finished with value: 0.9033506215996918 and parameters: {'SEED': 1036, 'WINDOW': 60, 'STRIDE': 15, 'c1_filters': 32, 'c2_filters': 64, 'c3_filters': 128, 'cnn_dropout': 0.009305236420321494, 'hidden_size': 32, 'num_layers': 3, 'rnn_dropout': 0.11967645219502371, 'rnn_type': 'GRU', 'bidirectional': False, 'init_scheme': 'kaiming_normal', 'lr': 2.2892331554376754e-05, 'weight_decay': 2.4546316528685736e-06, 'l1_lambda': 3.7432311184317826e-05, 'weight_max_norm': 1.9249998391601346}. Best is trial 0 with value: 0.9224502800789871.
[I 2025-11-17 15:40:18,321] Trial 2 pruned. 


...

Epoch  26/200 | Train: Loss=0.1026, F1 Score=0.9874 | Val: Loss=0.7122, F1 Score=0.9221
Training 200 epochs...
Epoch   1/200 | Train: Loss=1.0749, F1 Score=0.5902 | Val: Loss=1.0886, F1 Score=0.0756
Epoch   2/200 | Train: Loss=0.7629, F1 Score=0.7913 | Val: Loss=0.6681, F1 Score=0.8037
Epoch   3/200 | Train: Loss=0.5738, F1 Score=0.8448 | Val: Loss=0.6035, F1 Score=0.8179
Epoch   4/200 | Train: Loss=0.4663, F1 Score=0.8698 | Val: Loss=0.5384, F1 Score=0.8677


[I 2025-11-17 16:10:11,434] Trial 27 pruned. 


Epoch   5/200 | Train: Loss=0.3982, F1 Score=0.8845 | Val: Loss=0.6040, F1 Score=0.8362
Training 200 epochs...
Epoch   1/200 | Train: Loss=1.3418, F1 Score=0.5079 | Val: Loss=1.1781, F1 Score=0.0151
Epoch   2/200 | Train: Loss=1.0866, F1 Score=0.6907 | Val: Loss=0.6435, F1 Score=0.7806
Epoch   3/200 | Train: Loss=0.8702, F1 Score=0.7938 | Val: Loss=0.5832, F1 Score=0.7897
Epoch   4/200 | Train: Loss=0.7195, F1 Score=0.8352 | Val: Loss=0.4566, F1 Score=0.8792
Epoch   5/200 | Train: Loss=0.6365, F1 Score=0.8663 | Val: Loss=0.4787, F1 Score=0.8577
Epoch   6/200 | Train: Loss=0.5751, F1 Score=0.8867 | Val: Loss=0.4674, F1 Score=0.8784
Epoch   7/200 | Train: Loss=0.5114, F1 Score=0.9012 | Val: Loss=0.4765, F1 Score=0.8931
Epoch   8/200 | Train: Loss=0.4763, F1 Score=0.9106 | Val: Loss=0.5544, F1 Score=0.8644
Epoch   9/200 | Train: Loss=0.4354, F1 Score=0.9225 | Val: Loss=0.4641, F1 Score=0.9111
Epoch  10/200 | Train: Loss=0.4151, F1 Score=0.9272 | Val: Loss=0.5008, F1 Score=0.9019
Epoch  11

[I 2025-11-17 16:10:52,742] Trial 28 pruned. 


Epoch  12/200 | Train: Loss=0.3728, F1 Score=0.9385 | Val: Loss=0.5158, F1 Score=0.9062
Training 200 epochs...
Epoch   1/200 | Train: Loss=0.5921, F1 Score=0.8024 | Val: Loss=1.0257, F1 Score=0.5755
Epoch   2/200 | Train: Loss=0.3193, F1 Score=0.8817 | Val: Loss=0.7402, F1 Score=0.8275
Epoch   3/200 | Train: Loss=0.2200, F1 Score=0.9116 | Val: Loss=0.4780, F1 Score=0.9206
Epoch   4/200 | Train: Loss=0.1824, F1 Score=0.9221 | Val: Loss=0.5651, F1 Score=0.8598
Epoch   5/200 | Train: Loss=0.1692, F1 Score=0.9288 | Val: Loss=0.4946, F1 Score=0.8745
Epoch   6/200 | Train: Loss=0.1472, F1 Score=0.9386 | Val: Loss=0.5758, F1 Score=0.8761
Epoch   7/200 | Train: Loss=0.1434, F1 Score=0.9400 | Val: Loss=0.6059, F1 Score=0.8676
Epoch   8/200 | Train: Loss=0.1272, F1 Score=0.9430 | Val: Loss=0.6479, F1 Score=0.8776
Epoch   9/200 | Train: Loss=0.1086, F1 Score=0.9514 | Val: Loss=0.6648, F1 Score=0.9159
Epoch  10/200 | Train: Loss=0.1098, F1 Score=0.9576 | Val: Loss=0.5767, F1 Score=0.8914
Epoch  11

## Predict Public tests

In [ ]:
df_train_fold.head()

In [ ]:
def create_test_tta_windows(
    df,
    feature_cols,
    seq_length,
    stride,
    num_tta=3,
    noise_std=0.003
):
    """
    Create test windows + TTA augmentations.
    Compatible with build_sequences_with_ids output format.
    """

    all_sequences = []
    all_sample_ids = []

    for sample_id, group in df.groupby("sample_index"):
        X = group[feature_cols].values.astype(np.float32)
        total_steps = len(X)

        # standard slide across the sequence
        offsets = list(range(0, total_steps - seq_length + 1, stride))

        for offset in offsets:
            base_win = X[offset:offset + seq_length]

            # TTA 1 — original unchanged window
            all_sequences.append(base_win)
            all_sample_ids.append(sample_id)

            # TTA 2 — noisy versions
            for _ in range(num_tta):
                noise = np.random.normal(0, noise_std, base_win.shape).astype(np.float32)
                noisy_win = base_win + noise
                all_sequences.append(noisy_win)
                all_sample_ids.append(sample_id)

    return np.array(all_sequences, dtype=np.float32), np.array(all_sample_ids)


def collect_probabilities(model, data_loader, device, num_classes):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in data_loader:
            if isinstance(batch, (tuple, list)):
                inputs = batch[0]
            else:
                inputs = batch
            inputs = inputs.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
            probs.append(logits.softmax(dim=1).cpu().numpy())
    if not probs:
        return np.zeros((0, num_classes), dtype=np.float32)
    return np.concatenate(probs, axis=0)


def aggregate_probabilities(sample_ids, probs, num_classes):
    prob_cols = [f'prob_{i}' for i in range(num_classes)]
    df_probs = pd.DataFrame(probs, columns=prob_cols)
    df_probs['sample_index'] = sample_ids
    return df_probs.groupby('sample_index')[prob_cols].mean()


# --- Prepare final datasets ---
df_train_final = df_train_fold.copy()
df_val_final = df_val_fold.copy()
df_test_final = df_public_test.copy()

scaler_final = MinMaxScaler()
preProcess_with_2D_PE(df_train_final, scaler_final, fit_scaler=True)
preProcess_with_2D_PE(df_val_final, scaler_final, fit_scaler=False)
preProcess_with_2D_PE(df_test_final, scaler_final, fit_scaler=False)

final_feature_cols = [col for col in feature_cols if col in df_train_final.columns]
if not final_feature_cols:
    raise ValueError("No feature columns available for final training.")

df_train_final[final_feature_cols] = df_train_final[final_feature_cols].astype(np.float32)
df_val_final[final_feature_cols] = df_val_final[final_feature_cols].astype(np.float32)
df_test_final[final_feature_cols] = df_test_final[final_feature_cols].astype(np.float32)
df_train_final['label_encoded'] = df_train_final['label_encoded'].astype(np.int64)
df_val_final['label_encoded'] = df_val_final['label_encoded'].astype(np.int64)

window_final = study.best_params['WINDOW']
stride_final = study.best_params['STRIDE']


X_train_final, y_train_final, _ = build_sequences_with_ids(
    df_train_final, final_feature_cols, 'sample_index', window_final, stride_final, label_col='label_encoded'
)
X_val_final, y_val_final, val_sample_ids = build_sequences_with_ids(
    df_val_final, final_feature_cols, 'sample_index', window_final, stride_final, label_col='label_encoded'
)

X_test_final, test_sample_ids = create_test_tta_windows(
    df_test_final,
    final_feature_cols,
    window_final,
    stride_final,
    num_tta=3,
    noise_std=0.004
)


y_train_final = y_train_final.astype(np.int64)
y_val_final = y_val_final.astype(np.int64)

num_classes = 3
labels_in_train = np.unique(y_train_final)

class_weights_present = compute_class_weight('balanced', classes=labels_in_train, y=y_train_final)
weights_array = np.ones(num_classes, dtype=np.float32)
for label, weight in zip(labels_in_train, class_weights_present):
    weights_array[int(label)] = weight
weights_tensor = torch.tensor(weights_array, dtype=torch.float32, device=device)

train_ds_final = TensorDataset(torch.from_numpy(X_train_final), torch.from_numpy(y_train_final))
val_ds_final = TensorDataset(torch.from_numpy(X_val_final), torch.from_numpy(y_val_final))
test_ds_final = TensorDataset(torch.from_numpy(X_test_final))

train_loader_final = make_loader(train_ds_final, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader_final = make_loader(val_ds_final, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader_final = make_loader(test_ds_final, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

cnn_params = {
    'c1_filters': study.best_params['c1_filters'],
    'c2_filters': study.best_params['c2_filters'],
    'c3_filters': study.best_params['c3_filters'],
    'cnn_dropout': study.best_params['cnn_dropout']
}
rnn_params = {
    'hidden_size': study.best_params['hidden_size'],
    'num_layers': study.best_params['num_layers'],
    'rnn_dropout': study.best_params['rnn_dropout'],
    'bidirectional': study.best_params['bidirectional'],
    'rnn_type': study.best_params['rnn_type']
}

model_final = CSHN_HybridClassifier(
    cnn_params=cnn_params,
    rnn_params=rnn_params,
    num_raw_features=len(final_feature_cols),
    num_classes=num_classes
).to(device)
initialize_weights(model_final, init_scheme=study.best_params['init_scheme'])

criterion_final = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer_final = torch.optim.AdamW(model_final.parameters(), lr=study.best_params['lr'], weight_decay=study.best_params['weight_decay'])
scheduler_final = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_final, mode='max', factor=0.1, patience=10, min_lr=1e-7)
scaler_amp_final = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

os.makedirs("models", exist_ok=True)
os.makedirs("submissions", exist_ok=True)

model_final, history_final, best_val_f1 = fit(
    model=model_final,
    train_loader=train_loader_final,
    val_loader=val_loader_final,
    epochs=EPOCHS,
    criterion=criterion_final,
    optimizer=optimizer_final,
    scaler=scaler_amp_final,
    device=device,
    l1_lambda=study.best_params['l1_lambda'],
    l2_lambda=0,
    patience=PATIENCE,
    evaluation_metric="val_f1",
    mode='max',
    restore_best_weights=True,
    scheduler=scheduler_final,
    weight_max_norm=study.best_params['weight_max_norm']
)
print(f"Best validation F1 (window level): {best_val_f1:.4f}")

val_probs = collect_probabilities(model_final, val_loader_final, device, num_classes)
if val_probs.shape[0] != len(val_sample_ids):
    raise ValueError("Mismatch between validation probabilities and sample indices.")
val_probs_agg = aggregate_probabilities(val_sample_ids, val_probs, num_classes)
val_sample_targets = df_val_final.groupby('sample_index')['label_encoded'].first().loc[val_probs_agg.index].values
val_preds = val_probs_agg.values.argmax(axis=1)
val_accuracy = accuracy_score(val_sample_targets, val_preds)
val_f1_macro = f1_score(val_sample_targets, val_preds, average='macro')
print(f"Validation (per sample) - Accuracy: {val_accuracy:.4f}, Macro F1: {val_f1_macro:.4f}")

trial_number = study.best_trial.number
model_path = f"models/optuna_trial_{trial_number}_final_model.pt"
torch.save({
    "model_state_dict": model_final.state_dict(),
    "hyperparameters": study.best_params,
    "feature_cols": final_feature_cols,
    "scaler": scaler_final,
    "window": window_final,
    "stride": stride_final,
}, model_path)
print("Saved model:", model_path)

test_probs = collect_probabilities(model_final, test_loader_final, device, num_classes)
if test_probs.shape[0] != len(test_sample_ids):
    raise ValueError("Mismatch between test probabilities and sample indices.")
test_probs_agg = aggregate_probabilities(test_sample_ids, test_probs, num_classes)
test_preds = test_probs_agg.values.argmax(axis=1)
inverse_label_map = {v: k for k, v in label_map.items()}

test_sample_index_values = test_probs_agg.index.to_numpy()
if np.issubdtype(test_sample_index_values.dtype, np.number):
    test_sample_index_values = test_sample_index_values.astype(int)

submission_df = pd.DataFrame({
    "sample_index": test_sample_index_values,
    "label": [inverse_label_map[int(label)] for label in test_preds]
}).sort_values("sample_index").reset_index(drop=True)

submission_path = f"submissions/optuna_trial_{trial_number}_submission.csv"
submission_df.to_csv(submission_path, index=False)
print("Saved submission:", submission_path)

print(submission_df.head())

In [ ]:
submission_df.head(50)